# 🔥 Advanced · Video-LM (Qwen2-VL)

> 🔥 **Advanced / heavy lab.** Reason over a video with a video-language model (frames → tokens → LLM).
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **15–30 min (2B model)** including downloads. Built on the official **[Qwen2-VL](https://github.com/QwenLM/Qwen2-VL)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Directly relevant to **egocentric video understanding** (track C): natural-language Q&A over first-person clips.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — Qwen2-VL-2B inference, low fps/res | A100 40 GB — 2B fine-tune / 7B inference |
| **Storage** | model ~ 4 GB + a clip | video-instruction datasets 100 GB+; ~200 GB disk |
| **Time** | inference seconds–min / clip | LoRA fine-tune ~ hours–1 day |

**Full pipeline (scale-up):** LoRA SFT over a video-instruction dataset (multi-GPU).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q -U "transformers>=4.45" accelerate "qwen-vl-utils[decord]"

## 1 · Load Qwen2-VL-2B-Instruct

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
ckpt = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(ckpt, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(ckpt)

## 2 · Ask a question about your video
Upload `clip.mp4` (e.g. an egocentric cooking clip).

In [ ]:
from qwen_vl_utils import process_vision_info
messages = [{"role": "user", "content": [
    {"type": "video", "video": "clip.mp4", "max_pixels": 360*420, "fps": 1.0},
    {"type": "text", "text": "What is the person doing, step by step?"}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inp = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
out = model.generate(**inp, max_new_tokens=256)
trimmed = [o[len(i):] for i, o in zip(inp.input_ids, out)]
print(processor.batch_decode(trimmed, skip_special_tokens=True)[0])

## 3 · Fine-tune on video instructions (LoRA) — sketch
Wrap the model with PEFT LoRA and train on a video-instruction dataset (e.g. `lmms-lab/VideoChatGPT`), using a collator that runs `process_vision_info` per sample — same SFT loop as the VLM lab, with video inputs.

In [ ]:
from peft import LoraConfig, get_peft_model
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM"))
model.print_trainable_parameters()
# then: build a Dataset of {video, question, answer}, a collator calling process_vision_info,
# and a transformers/TRL Trainer exactly like the VLM lab. (Heavy — needs the video dataset.)

## Evaluate — video-understanding benchmarks
Video-LMs are scored on **Video-MME**, **MVBench**, **EgoSchema** (multiple-choice video-QA accuracy). For numbers, run **lmms-eval** on one suite; for a quick check, ask several questions about a known clip and verify the answers.

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/<lab>")`.

## How this links to tracks A–D
Video-language reasoning maps to:
- **C · Egocentric** first-person video Q&A and action understanding · **A · Human** describing / parsing human actions in video.

## Notes & next steps
- **Frames/fps & max_pixels** trade quality vs. memory — lower them if you OOM on a T4.
- Other video-LMs: **Video-LLaVA**, **VideoLLaMA-2**, **LLaVA-NeXT-Video**.
- Pair with the **VideoMAE** and **SAM 2** labs for a full egocentric-video stack.